##### Import Libraries:

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable

##### Adding Current Directory to System Path:

In [0]:
import os
import sys

In [0]:
current_dir = os.getcwd()
sys.path.append(current_dir)
sys.path

#### Generic Function:

In [0]:
class transformations:

    def dedup(self,df:DataFrame,dedup_columns:List,cdc:str):

        df = df.withColumn("dedupkey",concat(*dedup_columns))
        df = df.withColumn("dedupCounts",row_number().over(Window.partitionBy("dedupkey").orderBy(desc(cdc))))
        df = df.filter(col("dedupCounts")==1)
        df = df.drop("dedupkey","dedupCounts")
        return df
    
    def processing_timestamp(self,df:DataFrame):

        df = df.withColumn("processing_timestamp",current_timestamp())
        return df
    
    def upsert(self,df,key_cols,table,cdc):

        merge_condition = " AND ".join([f"src.{i} = trg.{i}" for i in key_cols])
        dlt_obj = DeltaTable.forName(spark,f"pysparkdbt.silver.{table}")
        dlt_obj.alias("trg").merge(df.alias("src"),merge_condition)\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
        return "Upsert Completed!"

#### CUSTOMER Transformations:

In [0]:
## Read the customer data from table
df_customer = spark.read.table("pysparkdbt.bronze.customers")
display(df_customer)

In [0]:
## Transformation 1: Collect the domain names of email ids of customer
df_customer = df_customer.withColumn("Domains", split(col("email"),"@")[1])
display(df_customer)

In [0]:
## Transformation 2: Clean the phone number, remove all the characters other then 0 to 9 numbers.
df_customer = df_customer.withColumn("phone_number", regexp_replace(col("phone_number"), r"[^0-9]",""))
display(df_customer)

In [0]:
## Transformation 3.1: Concat the first and last name and create a full name column
df_customer = df_customer.withColumn("full_name", concat("first_name",lit(" "),"last_name"))
display(df_customer)

In [0]:
## Transformation 3.2: Concat the first name and last name and create full name column with different method
df_customer = df_customer.withColumn("full_name", concat_ws(" ",col("first_name"),col("last_name")))
display(df_customer)

In [0]:
## Transformation 4: Drop columns first name and last name
df_customer = df_customer.drop("first_name","last_name")
display(df_customer)

###### DESCOPED: Call "Generic Function" made in another file called - "custom_utilities.py"

In [0]:
## Descoped!! We have migrated the generic function code in this notebook only, so now no need to import it.
from utilities.custom_utilities import transformations

In [0]:
## Generic Tranformation 1: De-duplication of a dataframe
df_cust = transformations()
df_cust_dedup = df_cust.dedup(df_customer,["customer_id"],"last_updated_timestamp")
display(df_cust_dedup)

In [0]:
## Generic Transformation 2: Create a new column called as processing time, which can be further used for upsert logic
df_process = transformations()
df_customers = df_process.processing_timestamp(df_cust_dedup)
display(df_customers)

#### Upsert Logic (Customers):

In [0]:
## Generic Transformation 3: Using upsert logic that is defined in generic function

if not spark.catalog.tableExists("pysparkdbt.silver.customers"):

    df_customers.write.format("delta")\
            .mode("append")\
            .saveAsTable("pysparkdbt.silver.customers")

else:

    transformations().upsert(df_customers,["customer_id"],"customers","last_updated_timestamp")


In [0]:
df = spark.read.table("pysparkdbt.silver.customers")
display(df.count())

#### DRIVERS Transformation:

In [0]:
## Read the driver data from table
df_drivers = spark.read.table("pysparkdbt.bronze.drivers")
display(df_drivers)

In [0]:
## Transformation 1 & 2: Concat the first and last name and create a full name column, and delete those two columns
df_drivers = df_drivers.withColumn("full_name", concat("first_name",lit(" "),"last_name"))
df_drivers = df_drivers.drop("first_name","last_name")

In [0]:
df_drivers = df_drivers.withColumn("phone_number", regexp_replace(col("phone_number"), r"[^0-9]",""))

In [0]:
driver_obj = transformations()
df_drivers = driver_obj.processing_timestamp(df_drivers)
df_drivers = driver_obj.dedup(df_drivers,["driver_id"],"last_updated_timestamp")
display(df_drivers.count())

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):
    df_drivers.write.format("delta")\
            .mode("append")\
            .saveAsTable("pysparkdbt.silver.drivers")
    print("New Table Created!!")

else:
    driver_obj.upsert(df_drivers,["driver_id"],"drivers","last_updated_timestamp")
    print("Upsert Completed!!")


final_count = spark.read.table("pysparkdbt.silver.drivers")
display(final_count.count())


#### LOCATIONS Transformations:

In [0]:
## Read the location data from table
df_locations = spark.read.table("pysparkdbt.bronze.locations")
display(df_locations)

In [0]:
df_locations = df_locations.withColumn("address", concat_ws(", ","city","state","country"))

In [0]:
location_obj = transformations()
df_locations = location_obj.processing_timestamp(df_locations)
df_locations = location_obj.dedup(df_locations,["location_id"],"last_updated_timestamp")
display(df_locations.count())

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.locations"):
    df_locations.write.format("delta")\
            .mode("append")\
            .saveAsTable("pysparkdbt.silver.locations")
    print("New Table Created!!")

else:
    location_obj.upsert(df_locations,["location_id"],"locations","last_updated_timestamp")
    print("Upsert Completed!!")


final_count = spark.read.table("pysparkdbt.silver.locations")
display(final_count.count())

#### PAYMENT Transformations:

In [0]:
## Read the payment data from table
df_payments = spark.read.table("pysparkdbt.bronze.payments")
display(df_payments)

In [0]:
df_payments = df_payments.withColumn("online_payment_status",
                                     when( ((col('payment_method')=='Card') & (col('payment_status')=='Completed')),'Online-Success')\
                                     .when( ((col('payment_method')=='Card') & (col('payment_status')=='Failed')),'Online-Failed')\
                                     .when( ((col('payment_method')=='Card') & (col('payment_status')=='Pending')),'Online-Pending')\
                                     .otherwise("Offline Payment")
                                     )

In [0]:
payment_obj = transformations()
df_payments = payment_obj.processing_timestamp(df_payments)
df_payments = payment_obj.dedup(df_payments,["payment_id"],"last_updated_timestamp")
display(df_payments.count())

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.payments"):
    df_payments.write.format("delta")\
            .mode("append")\
            .saveAsTable("pysparkdbt.silver.payments")
    print("New Table Created!!")

else:
    payment_obj.upsert(df_payments,["payment_id"],"payments","last_updated_timestamp")
    print("Upsert Completed!!")


final_count = spark.read.table("pysparkdbt.silver.payments")
display(final_count.count())

#### VEHICLES Transformations:

In [0]:
## Read the vehicles data from table
df_vehicles = spark.read.table("pysparkdbt.bronze.vehicles")
display(df_vehicles)

In [0]:
df_vehicles = df_vehicles.withColumn("make",upper("make"))

In [0]:
vehicles_obj = transformations()
df_vehicles = vehicles_obj.processing_timestamp(df_vehicles)
df_vehicles = vehicles_obj.dedup(df_vehicles,["vehicle_id"],"last_updated_timestamp")
display(df_vehicles.count())

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):
    df_vehicles.write.format("delta")\
            .mode("append")\
            .saveAsTable("pysparkdbt.silver.vehicles")
    print("New Table Created!!")

else:
    vehicles_obj.upsert(df_vehicles,["vehicle_id"],"vehicles","last_updated_timestamp")
    print("Upsert Completed!!")


final_count = spark.read.table("pysparkdbt.silver.vehicles")
display(final_count.count())